← [El problema](01-el-problema.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Reglas y hechos](03-sistema-experto-reglas.ipynb) →

# 02 · Dos formas de saber

Toda la inteligencia artificial de Reci pertenece a una de dos familias. No son
variantes de lo mismo: parten de ideas opuestas sobre qué significa que una
máquina "sepa" algo.



## Estado vigente del proyecto (actualizado el 13 de agosto de 2026)

Esta serie conserva explicaciones y resultados históricos, pero la referencia
operativa actual es la siguiente:

- El sistema experto tiene **193 reglas**, CF estilo MYCIN, meta-reglas,
  encadenamiento hacia adelante y hacia atrás. Su voto usa OpenAI como
  proveedor principal, con heurísticas OpenCV que refinan atributos.
- El modelo local que participa en la decisión es **MobileNetV2 TFLite
  float32**, corrida `run_20260721_2129`; clasifica solo `plastico | vidrio`.
  MobileNetV3-Large INT8 está archivado como respaldo y **no emite votos**.
- En 1.000 capturas OV3660/QVGA, V2 obtuvo **71,60 %** de exactitud y
  **71,25 %** de macro-F1; V3 INT8 obtuvo 57,10 % y 57,09 %. La validación
  histórica de 98,43 % no describe por sí sola el rendimiento del robot.
- La ESP32-CAM toma tres fotos: se suman los seis votos válidos de ambas
  fuentes. `desconocido` es abstención; un empate se resuelve con el proveedor.
  Si el proveedor se abstiene las tres veces, el modelo local necesita 3/3.

La documentación operativa es [`ia/vision-service/README.md`](../../ia/vision-service/README.md),
[`model/README.md`](../../ia/vision-service/model/README.md) y
[`PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md`](../../docs/PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md).


## La familia simbólica

El conocimiento se **escribe a mano**, como reglas explícitas:

```
SI el objeto es una botella de cerveza Y tiene tapa corona
ENTONCES es vidrio
```

Una persona que sabe del tema —un experto— traduce lo que sabe a un lenguaje
que la máquina puede seguir. La máquina no aprende nada: **aplica** lo que
alguien ya sabía.

Es la IA de los años 60 a 80. El caso célebre es **MYCIN**, un sistema de
Stanford que diagnosticaba infecciones sanguíneas a partir de reglas escritas
por médicos. Reci le hereda directamente su aritmética de certeza (documento
[04](04-factor-de-certeza.ipynb)).



## La familia conexionista

El conocimiento se **aprende de ejemplos**: el entrenamiento ajusta parámetros internos usando fotos etiquetadas. El experimento de agosto comparó MobileNetV2, EfficientNet-B0 y MobileNetV3-Large, pero el artefacto que participa hoy en la decisión es **MobileNetV2 TFLite float32**, corrida `run_20260721_2129`.

El 13 de agosto ambos artefactos disponibles se ejecutaron sobre 1.000 capturas OV3660/QVGA: V2 obtuvo 71,60 % de exactitud y V3-Large INT8 57,10 %. Por eso V3 queda archivado como respaldo sin voto. Esta comparación no sustituye una prueba reservada: sirve para elegir el artefacto de producción actual.


## Comparación

| | Simbólica (sistema experto) | Conexionista (red neuronal) |
| --- | --- | --- |
| Origen del conocimiento | Un humano lo escribe | Se induce de ejemplos |
| Necesita datos | No | Sí, muchos |
| Necesita un experto | Sí | No, pero sí etiquetas |
| ¿Explica su conclusión? | Sí, cita las reglas usadas | No, solo un número |
| Manejar un caso nuevo | Solo si estaba previsto | A veces sí, si se parece a lo visto |
| Corregir un error | Editar una regla | Reentrenar con más datos |
| Coste de mantenimiento | Crece con el nº de reglas | Crece con el nº de datos |



## Dónde falla cada una

Esta es la fila más importante de la tabla anterior, y merece desarrollarse.

**El sistema experto falla cuando el caso no está previsto.** Sus 193 reglas
cubren botellas de agua, cerveza, mocachino, Gatorade, Pony Malta… Si aparece
un envase que nadie anticipó, ninguna regla dispara y el sistema se abstiene.
Falla **por omisión**, y lo hace de forma visible: puedes preguntarle por qué y
te dirá qué reglas evaluó.

**La red neuronal falla cuando el caso se parece poco a lo entrenado.** Y aquí
está el problema serio: falla **con confianza**. En las capturas reales de la
ESP32-CAM, 17 de 25 errores de vidrio tuvieron confianza ≥ 0,90, y dos
exactamente 1,0000. El modelo estaba *seguro* mientras se equivocaba. No puedes
filtrar esos errores con un umbral, porque no se ven distintos de los aciertos.



## Por qué Reci usa las dos

Porque **fallan por razones distintas**, y eso es exactamente lo que hace útil
combinarlas.

Si dos sistemas se equivocan por la misma causa, juntarlos no ayuda: cuando uno
falla, el otro también. Pero si uno falla por falta de reglas y el otro por
falta de ejemplos parecidos, es poco probable que ambos se equivoquen en la
misma foto por el mismo motivo. Cada uno cubre el punto ciego del otro.

Esa es la apuesta del diseño híbrido de Reci. Y como toda apuesta, hay que
verificarla con datos —no darla por buena— que es de lo que trata el documento
[11](11-como-se-mide.ipynb).



## Un matiz sobre "explicable"

La capacidad de explicar no es un lujo académico aquí. Cuando el robot abre la
compuerta equivocada, alguien tiene que averiguar por qué. Con el sistema
experto se puede: `explanation.py` genera un reporte con las reglas que
dispararon y su contribución. Con la red neuronal no hay nada equivalente —
solo un número que dice 0,93 sin justificarlo.

Por eso el campo `rule_applied` viaja en la respuesta de la API:

```json
{ "material": "vidrio", "confidence": 0.95, "rule_applied": "VIDRIO · 3 regla(s) · CF 0.95" }
```

---

← [El problema](01-el-problema.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Reglas y hechos](03-sistema-experto-reglas.ipynb) →
